In [7]:
from pathlib import Path

import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from temgym_core.components import Lens, Detector, Plane, SigmoidAperture
from temgym_core.ray import Ray
from temgym_core.source import make_waist_divergence_rays
from temgym_core.gaussian import make_gaussian
from temgym_core.run import run_to_end_vmapped
from temgym_core.evaluate import evaluate_gaussians_gpu_kernel_wrapper
from temgym_core.plotting import plot_model_plotly
from temgym_core.microscope_model import MicroscopeModel
from temgym_core.utils import fibonacci_spiral

import jax
jax.config.update("jax_enable_x64", True)

In [8]:
model_path = Path("data/microscope_design_curves.npz")
model = MicroscopeModel.from_npz(model_path)

print(f"Loaded MicroscopeModel configured at {model.voltage/1000} kV.")
print(f"Modes: {list(model.modes.keys())}")
print("Condenser config values:", model.modes['spot'].control_values)
print("Magnification config values:", model.modes['mag'].control_values)

Loaded MicroscopeModel configured at 200.0 kV.
Modes: ['spot', 'mag']
Condenser config values: [1. 2. 3. 4. 5. 6. 7. 8.]
Magnification config values: [ 30000.  50000. 100000. 300000. 600000.]


In [9]:
spot_values = np.asarray(model.modes['spot'].control_values, dtype=float)
mag_values = np.asarray(model.modes['mag'].control_values, dtype=float)

aux = dict(getattr(model, 'auxiliary', {}) or {})
source_waist_m = float(aux.get('source_waist_m', 10e-9))
z_source = float(aux.get('z_source', 0.0))
grid_pixels = int(aux.get('gui_grid_pixels', 256))

sample_half_width_nm = float(aux.get('sample_half_width_nm', 3000.0))
detector_half_width_mm = 15.0
sample_half_width_m = sample_half_width_nm * 1e-9
detector_half_width_m = detector_half_width_mm * 1e-3

# Ray-fan plotting settings are notebook constants (not read from auxiliary metadata)
FAN_HALF_ANGLE_MRAD = 0.0
FAN_NUM_RAYS = 1

# Display settings
HORIZONTAL_SPACING = 0.2
BACKGROUND_COLOR = "white"
RAY_X_EXTENT_M = 15e-3
COMPONENT_LABEL_X_OFFSET = 0.01
SOURCE_MODE_INIT = 'Fibonacci'
FIBONACCI_SOURCE_COUNT = 1000
APERTURE_RADIUS_UM_INIT = 25.0
APERTURE_X_OFFSET_UM_INIT = 0.0
APERTURE_Y_OFFSET_UM_INIT = 0.0
APERTURE_EDGE_WIDTH_UM_INIT = 5.0
APERTURE_EDGE_WIDTH_UM_MIN = 0.5
APERTURE_EDGE_WIDTH_UM_MAX = 20.0
APERTURE_SHARPNESS = 1e11
APERTURE_T_LOW = 0.0
APERTURE_RADIUS_UM_MIN = 1.0
APERTURE_RADIUS_UM_MAX = 50.0
APERTURE_OFFSET_UM_MIN = -50.0
APERTURE_OFFSET_UM_MAX = 50.0
SAMPLE_GAIN_INIT = 1.0
DETECTOR_GAIN_INIT = 1.0

# Intensity reference: choose initial peak intensity I0 = 1.0 at the source waist
# and derive reference power from the implied Gaussian effective area.
I0_REF = 1.0
P_REF = I0_REF * (np.pi * source_waist_m * source_waist_m / 2.0)
P_REF = max(float(P_REF), 1e-30)

def em_to_lens(em):
    return Lens(z=float(em.z), focal_length=float(em.focal_length))

def make_fixed_detector(z_plane, half_width_m, shape=256):
    pixel = (2.0 * float(half_width_m)) / float(shape)
    return Detector(z=float(z_plane), pixel_size=(pixel, pixel), shape=(int(shape), int(shape)))

def run_to_end_fast(ray, components):
    return run_to_end_vmapped(ray, components)

def intensity_map(beam, detector):
    field = np.asarray(evaluate_gaussians_gpu_kernel_wrapper(beam, detector))
    intensity = np.abs(field) ** 2
    return intensity / P_REF

def image_axes_from_extent(extent, shape):
    ny, nx = shape
    x = np.linspace(extent[0], extent[1], nx)
    y = np.linspace(extent[2], extent[3], ny)
    return x, y

def to_plotly_1d(values):
    arr = np.asarray(values)
    if arr.ndim == 0:
        return [float(arr)]
    return arr.tolist()

def to_plotly_2d(values):
    return np.asarray(values).tolist()

def clone_trace(trace):
    return go.Figure(data=[trace]).data[0]

def make_input_beam(source_mode, theta0_from_waist):
    mode = str(source_mode).strip().lower()
    if mode.startswith('fibonacci'):
        n = int(FIBONACCI_SOURCE_COUNT)
        dx, dy = fibonacci_spiral(n, float(theta0_from_waist))
        dx = np.asarray(dx, dtype=float)
        dy = np.asarray(dy, dtype=float)
        amp = np.full((n,), 1.0 / max(n, 1), dtype=float)
        zeros = np.zeros((n,), dtype=float)
        infs = np.full((n,), np.inf, dtype=float)
        return make_gaussian(
            x=zeros, y=zeros, dx=dx, dy=dy, z=np.full((n,), z_source, dtype=float),
            voltage=np.full((n,), model.voltage, dtype=float),
            waist_x=np.full((n,), source_waist_m, dtype=float),
            waist_y=np.full((n,), source_waist_m, dtype=float),
            amp=amp,
            phase=zeros,
            rcurv_x=infs,
            rcurv_y=infs,
            wavelength_unit='m',
        )

    return make_gaussian(
        x=0.0, y=0.0, dx=0.0, dy=0.0, z=z_source,
        voltage=model.voltage,
        waist_x=source_waist_m,
        waist_y=source_waist_m,
        amp=1.0,
        phase=0.0,
        rcurv_x=np.inf,
        rcurv_y=np.inf,
        wavelength_unit='m',
    )

def build_state(
    spot_choice,
    magnification_choice,
    source_mode=SOURCE_MODE_INIT,
    aperture_radius_um=APERTURE_RADIUS_UM_INIT,
    aperture_x_offset_um=APERTURE_X_OFFSET_UM_INIT,
    aperture_y_offset_um=APERTURE_Y_OFFSET_UM_INIT,
    aperture_edge_width_um=APERTURE_EDGE_WIDTH_UM_INIT,
):
    spot_lenses = model.build_components('spot', float(spot_choice))
    mag_lenses = model.build_components('mag', float(magnification_choice))

    CL1_em, CL3_em, C_mini_em, Obj_prefield_em = spot_lenses[:4]
    Obj_post_em, IL1_em, IL2_em, IL3_em, PL1_em = mag_lenses[4:]

    z_sample = float(aux.get('z_sample', Obj_prefield_em.z + abs(Obj_prefield_em.focal_length)))
    if 'z_detector' in aux:
        z_detector = float(aux['z_detector'])
    else:
        d_pl1_to_detector = float(aux.get('d_pl1_to_detector_m', 0.325))
        z_detector = float(PL1_em.z + d_pl1_to_detector)

    sample_plane = Plane(z=z_sample)
    detector_plane = Plane(z=z_detector)

    CL1 = em_to_lens(CL1_em)
    CL3 = em_to_lens(CL3_em)
    C_mini = em_to_lens(C_mini_em)
    Obj_prefield = em_to_lens(Obj_prefield_em)
    Obj_post = em_to_lens(Obj_post_em)
    IL1_lens = em_to_lens(IL1_em)
    IL2_lens = em_to_lens(IL2_em)
    IL3_lens = em_to_lens(IL3_em)
    PL1_lens = em_to_lens(PL1_em)

    z_aperture = float(aux.get('z_aperture', 0.5 * (CL3_em.z + C_mini_em.z)))
    aperture = SigmoidAperture(
        z=z_aperture,
        radius=float(aperture_radius_um) * 1e-6,
        x0=float(aperture_x_offset_um) * 1e-6,
        y0=float(aperture_y_offset_um) * 1e-6,
        edge_width=float(aperture_edge_width_um) * 1e-6,
        sharpness=APERTURE_SHARPNESS,
        t_low=APERTURE_T_LOW,
        t_high=1.0,
    )
    aperture_plot = Plane(z=z_aperture)

    components_to_sample = [CL1, CL3, aperture, C_mini, Obj_prefield, sample_plane]
    components_to_detector = [
        CL1, CL3, aperture, C_mini, Obj_prefield, sample_plane,
        Obj_post, IL1_lens, IL2_lens, IL3_lens, PL1_lens, detector_plane,
    ]

    components_for_plot = [
        CL1_em, CL3_em, aperture_plot, C_mini_em, Obj_prefield_em, sample_plane,
        Obj_post_em, IL1_em, IL2_em, IL3_em, PL1_em,
        make_fixed_detector(z_detector, detector_half_width_m, shape=grid_pixels),
    ]

    component_labels = [
        "CL1", "CL3", "Aperture", "C_mini", "Obj_prefield", "Sample",
        "Obj_post", "IL1", "IL2", "IL3", "PL1", "Detector",
    ]

    source_basis = make_waist_divergence_rays(
        waist=source_waist_m,
        voltage=model.voltage,
        z=z_source,
        x0=0.0,
        y0=0.0,
    )
    theta0_from_waist = float(np.max(np.abs(np.asarray(source_basis.dx))))
    beam_in = make_input_beam(source_mode, theta0_from_waist)

    beam_at_sample = run_to_end_fast(beam_in, components_to_sample)
    sample_detector = make_fixed_detector(z_sample, sample_half_width_m, shape=grid_pixels)
    sample_image = intensity_map(beam_at_sample, sample_detector)
    sample_extent_nm = (
        -sample_half_width_nm,
        sample_half_width_nm,
        -sample_half_width_nm,
        sample_half_width_nm,
    )

    beam_at_detector = run_to_end_fast(beam_in, components_to_detector)
    detector_detector = make_fixed_detector(z_detector, detector_half_width_m, shape=grid_pixels)
    detector_image = intensity_map(beam_at_detector, detector_detector)
    detector_extent_mm = (
        -detector_half_width_mm,
        detector_half_width_mm,
        -detector_half_width_mm,
        detector_half_width_mm,
    )

    theta0 = max(theta0_from_waist, FAN_HALF_ANGLE_MRAD * 1e-3)
    dx_fan = np.linspace(-theta0, theta0, FAN_NUM_RAYS)
    fan_rays = Ray(
        x=np.zeros_like(dx_fan),
        y=np.zeros_like(dx_fan),
        dx=dx_fan,
        dy=np.zeros_like(dx_fan),
        z=np.full_like(dx_fan, z_source),
        pathlength=np.zeros_like(dx_fan),
    )

    return {
        'sample_image': sample_image,
        'sample_extent_nm': sample_extent_nm,
        'detector_image': detector_image,
        'detector_extent_mm': detector_extent_mm,
        'components_for_plot': components_for_plot,
        'component_labels': component_labels,
        'fan_rays': fan_rays,
        'source_basis': source_basis,
        'source_mode': str(source_mode),
        'aperture_radius_um': float(aperture_radius_um),
        'aperture_x_offset_um': float(aperture_x_offset_um),
        'aperture_y_offset_um': float(aperture_y_offset_um),
        'aperture_edge_width_um': float(aperture_edge_width_um),
        'theta0_mrad': theta0 * 1e3,
    }

def build_ray_figure(state):
    kwargs = dict(
        rays=state['fan_rays'],
        solution_rays=state['source_basis'],
        component_labels=state['component_labels'],
        include_input_rays=True,
        band_mode='lines',
        ray_coordinate='x_rot',
        width=700,
        height=620,
    )
    try:
        return plot_model_plotly(state['components_for_plot'], show_component_labels=False, **kwargs)
    except TypeError:
        return plot_model_plotly(state['components_for_plot'], **kwargs)

def component_label_annotations(state, label_x_paper):
    labels = state['component_labels']
    components = state['components_for_plot']
    annotations = []
    for i, (name, comp) in enumerate(zip(labels, components)):
        if not hasattr(comp, 'z'):
            continue
        zc = float(getattr(comp, 'z'))
        annotations.append(dict(
            x=label_x_paper,
            y=zc,
            xref='paper',
            yref='y3',
            text=f"{name} (z={zc:.4f} m)" ,
            showarrow=False,
            xanchor='left',
            yanchor='middle',
            align='left',
            font=dict(size=11, color='black'),
        ))
    return annotations

initial_spot = float(spot_values[0])
initial_mag = float(mag_values[0])
state = build_state(
    initial_spot,
    initial_mag,
    source_mode=SOURCE_MODE_INIT,
    aperture_radius_um=APERTURE_RADIUS_UM_INIT,
    aperture_x_offset_um=APERTURE_X_OFFSET_UM_INIT,
    aperture_y_offset_um=APERTURE_Y_OFFSET_UM_INIT,
    aperture_edge_width_um=APERTURE_EDGE_WIDTH_UM_INIT,
)
ray_fig = build_ray_figure(state)

sx, sy = image_axes_from_extent(state['sample_extent_nm'], state['sample_image'].shape)
dx, dy = image_axes_from_extent(state['detector_extent_mm'], state['detector_image'].shape)

# Lock color scales to the initial reference-normalized maps
sample_cmax = max(float(np.max(state['sample_image'])), 1e-30)
detector_cmax = max(float(np.max(state['detector_image'])), 1e-30)

fig = go.FigureWidget(
    make_subplots(
        rows=1,
        cols=3,
        column_widths=[0.30, 0.30, 0.40],
        horizontal_spacing=HORIZONTAL_SPACING,
        subplot_titles=(
            'Gaussian intensity at sample',
            'Gaussian intensity at detector',
            f"Ray diagram (x_rot), fan=±{state['theta0_mrad']:.2f} mrad",
        ),
    )
)

fig.add_trace(
    go.Heatmap(
        z=to_plotly_2d(state['sample_image'] * SAMPLE_GAIN_INIT),
        x=to_plotly_1d(sx),
        y=to_plotly_1d(sy),
        colorscale='Inferno',
        zmin=0.0,
        zmax=sample_cmax,
        colorbar=dict(
            title='I / P_ref [1/m²]',
            x=0.26,
            y=0.5,
            yanchor='middle',
            len=1.0,
            lenmode='fraction',
        ),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Heatmap(
        z=to_plotly_2d(state['detector_image'] * DETECTOR_GAIN_INIT),
        x=to_plotly_1d(dx),
        y=to_plotly_1d(dy),
        colorscale='Viridis',
        zmin=0.0,
        zmax=detector_cmax,
        colorbar=dict(
            title='I / P_ref [1/m²]',
            x=0.60,
            y=0.5,
            yanchor='middle',
            len=1.0,
            lenmode='fraction',
        ),
    ),
    row=1,
    col=2,
)

RAY_TRACE_OFFSET = 2
RAY_TRACE_COUNT = 0
for trace in ray_fig.data:
    if getattr(trace, 'mode', None) == 'text':
        continue
    fig.add_trace(clone_trace(trace), row=1, col=3)
    RAY_TRACE_COUNT += 1

fig.update_xaxes(
    title_text='x [nm]',
    row=1,
    col=1,
    constrain='domain',
    automargin=True,
)
fig.update_yaxes(
    title_text='y [nm]',
    row=1,
    col=1,
    scaleanchor='x',
    scaleratio=1,
    constrain='domain',
    automargin=True,
)
fig.update_xaxes(
    title_text='x [mm]',
    row=1,
    col=2,
    constrain='domain',
    automargin=True,
)
fig.update_yaxes(
    title_text='y [mm]',
    row=1,
    col=2,
    scaleanchor='x2',
    scaleratio=1,
    constrain='domain',
    automargin=True,
)
fig.update_xaxes(
    title_text='x_rot [m]',
    row=1,
    col=3,
    range=[-RAY_X_EXTENT_M, RAY_X_EXTENT_M],
    autorange=False,
)
fig.update_yaxes(title_text='z [m]', row=1, col=3, autorange='reversed')
fig.update_layout(
    height=620,
    width=1650,
    margin=dict(l=30, r=260, t=70, b=20),
    uirevision='keep-zoom',
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
)
component_label_x = float(fig.layout.xaxis3.domain[1]) + COMPONENT_LABEL_X_OFFSET
base_annotations = [a.to_plotly_json() for a in fig.layout.annotations[:3]]
fig.layout.annotations = tuple(base_annotations + component_label_annotations(state, component_label_x))

spot_slider = widgets.SelectionSlider(
    options=[float(v) for v in spot_values],
    value=initial_spot,
    description='Spot',
    continuous_update=False,
)
mag_slider = widgets.SelectionSlider(
    options=[float(v) for v in mag_values],
    value=initial_mag,
    description='Magnification',
    continuous_update=False,
)
source_mode_slider = widgets.Dropdown(
    options=['Single Gaussian', 'Fibonacci'],
    value=SOURCE_MODE_INIT,
    description='Source',
)
aperture_radius_slider = widgets.FloatSlider(
    value=APERTURE_RADIUS_UM_INIT,
    min=APERTURE_RADIUS_UM_MIN,
    max=APERTURE_RADIUS_UM_MAX,
    step=0.5,
    description='Aperture r [µm]',
    continuous_update=False,
)
aperture_x_slider = widgets.FloatSlider(
    value=APERTURE_X_OFFSET_UM_INIT,
    min=APERTURE_OFFSET_UM_MIN,
    max=APERTURE_OFFSET_UM_MAX,
    step=0.5,
    description='Aperture x [µm]',
    continuous_update=False,
)
aperture_y_slider = widgets.FloatSlider(
    value=APERTURE_Y_OFFSET_UM_INIT,
    min=APERTURE_OFFSET_UM_MIN,
    max=APERTURE_OFFSET_UM_MAX,
    step=0.5,
    description='Aperture y [µm]',
    continuous_update=False,
)
aperture_edge_slider = widgets.FloatSlider(
    value=APERTURE_EDGE_WIDTH_UM_INIT,
    min=APERTURE_EDGE_WIDTH_UM_MIN,
    max=APERTURE_EDGE_WIDTH_UM_MAX,
    step=0.5,
    description='Aperture edge [µm]',
    continuous_update=False,
)
sample_gain_slider = widgets.FloatLogSlider(
    value=SAMPLE_GAIN_INIT,
    base=10,
    min=-3,
    max=3,
    step=0.1,
    description='Sample gain',
    continuous_update=False,
)
detector_gain_slider = widgets.FloatLogSlider(
    value=DETECTOR_GAIN_INIT,
    base=10,
    min=-3,
    max=3,
    step=0.1,
    description='Detector gain',
    continuous_update=False,
)

def update_plot(*_):
    state = build_state(
        float(spot_slider.value),
        float(mag_slider.value),
        source_mode=source_mode_slider.value,
        aperture_radius_um=float(aperture_radius_slider.value),
        aperture_x_offset_um=float(aperture_x_slider.value),
        aperture_y_offset_um=float(aperture_y_slider.value),
        aperture_edge_width_um=float(aperture_edge_slider.value),
    )
    ray_fig = build_ray_figure(state)

    sx, sy = image_axes_from_extent(state['sample_extent_nm'], state['sample_image'].shape)
    dx, dy = image_axes_from_extent(state['detector_extent_mm'], state['detector_image'].shape)

    with fig.batch_update():
        fig.data[0].z = to_plotly_2d(state['sample_image'] * float(sample_gain_slider.value))
        fig.data[0].x = to_plotly_1d(sx)
        fig.data[0].y = to_plotly_1d(sy)

        fig.data[1].z = to_plotly_2d(state['detector_image'] * float(detector_gain_slider.value))
        fig.data[1].x = to_plotly_1d(dx)
        fig.data[1].y = to_plotly_1d(dy)

        ray_traces = [t for t in ray_fig.data if getattr(t, 'mode', None) != 'text']
        trace_count = min(RAY_TRACE_COUNT, len(ray_traces))
        for i in range(trace_count):
            fig.data[RAY_TRACE_OFFSET + i].x = to_plotly_1d(ray_traces[i].x)
            fig.data[RAY_TRACE_OFFSET + i].y = to_plotly_1d(ray_traces[i].y)

        fig.layout.annotations[2].text = (
            f"Ray diagram (x_rot), spot={spot_slider.value:.3g}, "
            f"mag={mag_slider.value:.3g}, src={state['source_mode']}, "
            f"ap_r={state['aperture_radius_um']:.1f} µm, "
            f"ap_xy=({state['aperture_x_offset_um']:.1f}, {state['aperture_y_offset_um']:.1f}) µm, "
            f"ap_edge={state['aperture_edge_width_um']:.1f} µm, "
            f"fan=±{state['theta0_mrad']:.2f} mrad"
        )
        base_annotations[2]['text'] = fig.layout.annotations[2].text
        fig.layout.annotations = tuple(base_annotations + component_label_annotations(state, component_label_x))

        fig.layout.plot_bgcolor = BACKGROUND_COLOR
        fig.layout.paper_bgcolor = BACKGROUND_COLOR

spot_slider.observe(update_plot, names='value')
mag_slider.observe(update_plot, names='value')
source_mode_slider.observe(update_plot, names='value')
aperture_radius_slider.observe(update_plot, names='value')
aperture_x_slider.observe(update_plot, names='value')
aperture_y_slider.observe(update_plot, names='value')
aperture_edge_slider.observe(update_plot, names='value')
sample_gain_slider.observe(update_plot, names='value')
detector_gain_slider.observe(update_plot, names='value')

display(widgets.VBox([
    fig,
    spot_slider,
    mag_slider,
    source_mode_slider,
    aperture_radius_slider,
    aperture_x_slider,
    aperture_y_slider,
    aperture_edge_slider,
    sample_gain_slider,
    detector_gain_slider,
]))

    'data': [{'colorbar': {'len': 1.0,
                           'lenmode': 'fr…